# 4.1 降水滴谱分布 (DSD) 调节器

**学习目标**
- 理解指数分布和Gamma分布的数学表达式
- 观察降雨率对滴谱分布的影响
- 掌握不同滴谱参数对雷达变量的影响
- 比较不同分布模型的特点

**参考文献**：Ryzhkov & Zrnic, Chapter 4, pp. 351-395

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### Marshall-Palmer 指数分布

$$N(D) = N_0 \exp(-\Lambda D)$$

其中 $N_0 = 8 \times 10^6$ m⁻⁴，$\Lambda = 4.1 R^{-0.21}$ mm⁻¹，$R$ 是降雨率(mm/h)。

### Gamma 分布

$$N(D) = N_0 D^\mu \exp(-\Lambda D)$$

其中 $\mu$ 是形状参数，通常取0-6之间的值。

In [ ]:
def marshall_palmer(R, D):
    """
    Marshall-Palmer 滴谱分布
    R: 降雨率 (mm/h)
    D: 直径 (mm)
    """
    N0 = 8e3  # mm^-1 m^-3 (原文用m^-4，这里用mm^-1 m^-3)
    Lambda = 4.1 * R**(-0.21)  # mm^-1
    return N0 * np.exp(-Lambda * D)

def gamma_dsd(D, N0=5e3, Lambda=3.0, mu=2):
    """
    Gamma 滴谱分布
    N0: 截距参数
    Lambda: 斜率参数
    mu: 形状参数
    """
    return N0 * D**mu * np.exp(-Lambda * D)

def calculate_rainfall_from_dsd(D, N_D, dD):
    """
    从滴谱计算降雨率
    R = sum(V(D) * pi/6 * D^3 * rho_w * N(D)) * dD
    """
    # 终端速度 (Atlas-Livesey简化)
    V = 9.65 * (1 - np.exp(-D / 0.9))  # m/s
    # 质量通量
    rho_w = 1000  # kg/m^3
    mass_flux = V * (np.pi/6) * (D*1e-3)**3 * rho_w * N_D * 1e3  # 转换为L/h单位
    R_calc = np.trapz(mass_flux, D)  # mm/h
    return R_calc

def calculate_dsd_moments(D, N_D, dD):
    """计算滴谱的各阶矩"""
    D_m = D * 1e-3  # 转换为m
    M0 = np.trapz(N_D * 1e3, D)  # 浓度 (m^-3)
    M3 = np.trapz(N_D * 1e3 * D_m**3, D)  # 反射率矩
    M4 = np.trapz(N_D * 1e3 * D_m**4, D)
    M6 = np.trapz(N_D * 1e3 * D_m**6, D)
    
    # 等效直径
    D0 = (M3 / M0)**(1/3) if M0 > 0 else 0
    
    # 反射率 Z (mm^6/m^3)
    Z = M6 * 1e18  # 转换为 mm^6/m^3
    
    return {'M0': M0, 'M3': M3, 'M4': M4, 'M6': M6, 'D0': D0*1000, 'Z': Z}

## 2. 滴谱分布可视化

In [ ]:
def plot_dsd_explorer(R=10.0, dsd_model='MP', N0_param=8e3, Lambda_param=None, mu_param=2):
    """交互式滴谱分布浏览器"""
    
    # 直径范围 (mm)
    D = np.linspace(0.1, 8, 300)
    
    if dsd_model == 'MP':
        N_D = marshall_palmer(R, D)
        title = f'Marshall-Palmer 分布 (R={R} mm/h)'
    else:  # Gamma
        if Lambda_param is None:
            Lambda_param = 4.1 * R**(-0.21)
        N_D = gamma_dsd(D, N0=N0_param, Lambda=Lambda_param, mu=mu_param)
        title = f'Gamma 分布 (μ={mu_param}, Λ={Lambda_param:.2f})'
    
    # 计算统计量
    dD = D[1] - D[0]
    moments = calculate_dsd_moments(D, N_D, dD)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: 滴谱分布线性坐标
    ax1 = axes[0, 0]
    ax1.plot(D, N_D, 'b-', linewidth=2)
    ax1.fill_between(D, N_D, alpha=0.3)
    ax1.set_xlabel('直径 D (mm)', fontsize=11)
    ax1.set_ylabel('N(D) (mm⁻¹ m⁻³)', fontsize=11)
    ax1.set_title(title, fontsize=12)
    ax1.set_xlim(0, 8)
    ax1.grid(True, alpha=0.3)
    
    # 子图2: 半对数坐标
    ax2 = axes[0, 1]
    ax2.semilogy(D, N_D, 'r-', linewidth=2)
    ax2.set_xlabel('直径 D (mm)', fontsize=11)
    ax2.set_ylabel('N(D) (mm⁻¹ m⁻³)', fontsize=11)
    ax2.set_title('半对数坐标滴谱', fontsize=12)
    ax2.set_xlim(0, 8)
    ax2.set_ylim(1, 1e6)
    ax2.grid(True, alpha=0.3)
    
    # 子图3: D^6加权（对Z的贡献）
    ax3 = axes[1, 0]
    D6_weighted = N_D * D**6
    ax3.plot(D, D6_weighted / D6_weighted.max(), 'g-', linewidth=2)
    # 标记众数和主导直径
    mode_D = D[np.argmax(N_D)]
    peak_Z_D = D[np.argmax(D6_weighted)]
    ax3.axvline(x=mode_D, color='b', linestyle='--', label=f'滴谱众数: {mode_D:.2f} mm')
    ax3.axvline(x=peak_Z_D, color='r', linestyle='--', label=f'Z贡献峰: {peak_Z_D:.2f} mm')
    ax3.set_xlabel('直径 D (mm)', fontsize=11)
    ax3.set_ylabel('归一化贡献', fontsize=11)
    ax3.set_title('D⁶ 加权滴谱（反映对雷达反射率的贡献）', fontsize=12)
    ax3.legend()
    ax3.set_xlim(0, 8)
    ax3.grid(True, alpha=0.3)
    
    # 子图4: 不同降雨率的对比
    ax4 = axes[1, 1]
    R_values = [1, 5, 10, 20, 50]
    colors = plt.cm.rainbow(np.linspace(0, 1, len(R_values)))
    for R_val, c in zip(R_values, colors):
        N_val = marshall_palmer(R_val, D)
        ax4.semilogy(D, N_val, color=c, linewidth=2, label=f'R={R_val} mm/h')
    ax4.set_xlabel('直径 D (mm)', fontsize=11)
    ax4.set_ylabel('N(D) (mm⁻¹ m⁻³)', fontsize=11)
    ax4.set_title('不同降雨率的Marshall-Palmer分布', fontsize=12)
    ax4.legend(loc='upper right')
    ax4.set_xlim(0, 8)
    ax4.set_ylim(1, 1e6)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== 滴谱统计量 ===")
    print(f"模型: {'Marshall-Palmer' if dsd_model == 'MP' else 'Gamma'}")
    print(f"输入降雨率: {R} mm/h")
    print(f"浓度 N0: {moments['M0']:.2e} m⁻³")
    print(f"中值直径 D0: {moments['D0']:.2f} mm")
    print(f"反射率 Z: {10*np.log10(moments['Z']+1e-10):.1f} dBZ")
    print(f"Z贡献峰直径: {peak_Z_D:.2f} mm")

In [ ]:
interact_dsd = interact(plot_dsd_explorer,
    R=widgets.FloatSlider(min=0.5, max=100, step=0.5, value=10.0, 
                         description='降雨率 (mm/h)'),
    dsd_model=widgets.Dropdown(options=[('Marshall-Palmer', 'MP'), ('Gamma', 'Gamma')], 
                              value='MP', description='分布模型'),
    N0_param=widgets.FloatText(value=8e3, description='N0 (mm⁻¹ m⁻³)'),
    Lambda_param=widgets.FloatText(value=None, description='Λ (mm⁻¹)'),
    mu_param=widgets.FloatSlider(min=0, max=6, step=0.5, value=2.0, description='μ (形状参数)')
)

## 3. 练习题

1. **分布形态**：Marshall-Palmer分布中，Λ如何随降雨率变化？这对滴谱形状有何影响？

2. **主导直径**：对于R=10mm/h的降雨，对反射率贡献最大的滴粒直径约是多少？

3. **Gamma分布**：形状参数μ的作用是什么？μ=0时Gamma分布退化成什么？

4. **D0与降雨率**：中值直径D0与降雨率R之间有什么关系？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 4, Springer.
- Marshall, J.S., and W.M.K. Palmer, 1948: The distribution of raindrops with size. *J. Atmos. Sci.*, 5, 165-166.